**Last modified on**: 07/05/2026

**Author**: Onur Serçinoğlu

**Update Logs**:

07/05/2026: Initial version. Replaces the older `4_blast_msa_phylogeny.ipynb`.

**Credits**:

This Jupyter notebook accompanies the BLAST lecture prepared for BENG451/BSB511 at Gebze Technical University. Key references:
- Altschul SF et al. (1990). Basic local alignment search tool. *J Mol Biol* 215:403–410.
- Altschul SF et al. (1997). Gapped BLAST and PSI-BLAST: a new generation of protein database search programs. *Nucleic Acids Res* 25:3389–3402.
- Karlin S, Altschul SF (1990). Methods for assessing the statistical significance of molecular sequence features by using general scoring schemes. *PNAS* 87:2264–2268.
- Madeira F et al. (2024). The EMBL-EBI Job Dispatcher sequence analysis tools framework in 2024. *Nucleic Acids Res* 52:W521–W525.

# Database search with BLAST

BLAST is the workhorse of homology search. Given a query sequence and a database, it answers the question *"which database sequences look enough like my query that I should be paying attention?"*. It does this fast, with a heuristic *word-seed* + *gapped extension* algorithm, and reports a calibrated **E-value** that tells you how surprised to be by each hit.

In this notebook we will:

1. Fetch a query sequence (human β-globin) from NCBI.
2. Run **`blastp`** against SwissProt **via the NCBI web service** (`Bio.Blast.NCBIWWW.qblast`).
3. Parse the XML output, build a hits table, and study the E-value / bit-score distribution.
4. Apply a sensible filter and write a multi-FASTA of selected hits — the input file for `5_2_msa.ipynb`.
5. Run **PSI-BLAST** (one call, web service) to see what iterative-profile search adds at the twilight zone.

The output of this notebook (`blast_work/selected_globins.fa`) is the input to `5_2_msa.ipynb`. The selection rule is deliberately tuned so leghemoglobin (a plant globin) is pulled in — that gives `5_3_phylogeny.ipynb` a natural outgroup.

For an interactive view of word seeding and HSP extension, see the `fasta-blast.html` widget. For the substitution matrix that shapes BLAST's score, see `scoring-matrices.html`.

In [ ]:
import os

# ─── Adjustable paths ────────────────────────────────────────────────────────
WORK_DIR     = "blast_work"
QUERY_ACC    = "P68871"           # human β-globin (UniProt accession)
DATABASE     = "swissprot"        # curated; fast; ideal teaching target
EVALUE_CUT   = 1e-3               # selection threshold for downstream MSA
HITLIST_SIZE = 100                # plenty of vertebrate orthologs; see PSI-BLAST below for divergent ones
MAX_FOR_MSA  = 20                 # cap on sequences passed downstream — keeps the MSA + tree readable

# NCBI politeness — set your contact email here. NCBI tracks this via Entrez
# headers and uses it to contact you if your traffic looks abusive (and to
# whitelist legitimate research traffic).
NCBI_EMAIL   = "your.name@example.org"   # ← REPLACE THIS WITH YOUR EMAIL

# Cache freshness window (hours). If the cached BLAST XML is newer than this,
# we refuse to re-submit. This is stricter than a "no-cache" approach and is
# the pattern the term project also uses.
CACHE_MAX_AGE_HOURS = 24
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Working directory: {os.path.abspath(WORK_DIR)}")
print(f"Query:             {QUERY_ACC}  (database: {DATABASE})")

---

In [ ]:
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.SeqRecord import SeqRecord

Entrez.email = NCBI_EMAIL

## 1. The E-value, briefly

BLAST scores every alignment with a substitution matrix (default BLOSUM62). The raw score $S$ is then converted to a **bit score** that is database-size independent:

$$S' = \frac{\lambda S - \ln K}{\ln 2}$$

The **E-value** answers "how many alignments at least this good would I expect *by chance* in a database of this size?":

$$E = K\, m\, n\, e^{-\lambda S}\quad\equiv\quad m\, n\, 2^{-S'}$$

where $m$ is query length, $n$ is total database length, and $\lambda, K$ are matrix-specific constants from Karlin–Altschul statistics.

**Rules of thumb:**
- $E < 10^{-50}$: essentially identical proteins.
- $E < 10^{-3}$: confident homology.
- $10^{-3} < E < 1$: "twilight zone" — could be true homologs or chance hits.
- $E > 1$: noise.

We will use $E < 10^{-3}$ as our selection cutoff (loose enough to pull in genuinely distant globins like plant leghemoglobin).

In [ ]:
# Fetch the query protein sequence from UniProt (cached on disk).
import urllib.request

query_path = os.path.join(WORK_DIR, f"{QUERY_ACC}.fasta")
if os.path.exists(query_path):
    print(f"Cache hit: {query_path}")
else:
    url = f"https://rest.uniprot.org/uniprotkb/{QUERY_ACC}.fasta"
    req = urllib.request.Request(url, headers={"User-Agent": f"gtubeng/5_1_blast {NCBI_EMAIL}"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        with open(query_path, "wb") as g:
            g.write(resp.read())
    print(f"Fetched: {query_path}")

query_record = SeqIO.read(query_path, "fasta")
print(f"\nQuery: {query_record.id}")
print(f"  {query_record.description[:100]}")
print(f"  length = {len(query_record.seq)} residues")

## 2. Polite, cached web BLAST

A single student running `qblast` once a semester is fine. The trouble starts when fourteen project teams hit the same NCBI endpoint from the same university IP block on the same evening. The defence is **caching**:

- After a successful `qblast`, write the XML output to disk.
- Before submitting again, check the cache. If the file is younger than `CACHE_MAX_AGE_HOURS`, refuse to re-submit.
- This protects NCBI from accidental re-runs and protects you from waiting in their queue twice.

`NCBIWWW.qblast` does **not** take an `email=` keyword; the Entrez email you set above is what NCBI sees in the HTTP headers (and what they will use to contact you if there's a problem).

In [ ]:
def cached_qblast(program, database, sequence, out_xml,
                  expect=10, hitlist_size=50, service="plain", **kwargs):
    """Run NCBIWWW.qblast, but cache aggressively.

    If `out_xml` exists and is younger than `CACHE_MAX_AGE_HOURS`, return the
    cached path without contacting NCBI. Otherwise submit, save, and return.
    """
    if os.path.exists(out_xml):
        age_h = (time.time() - os.path.getmtime(out_xml)) / 3600
        if age_h < CACHE_MAX_AGE_HOURS:
            print(f"  Cache hit: {out_xml}  (age {age_h:.1f} h)")
            return out_xml
        print(f"  Cache stale ({age_h:.1f} h > {CACHE_MAX_AGE_HOURS} h) — refreshing")

    print(f"  Submitting BLAST: {program} vs {database} ({len(sequence)} aa) ...")
    t0 = time.time()
    handle = NCBIWWW.qblast(
        program       = program,
        database      = database,
        sequence      = str(sequence),
        expect        = expect,
        hitlist_size  = hitlist_size,
        format_type   = "XML",
        service       = service,
        **kwargs,
    )
    xml_text = handle.read()
    handle.close()
    with open(out_xml, "w") as g:
        g.write(xml_text)
    print(f"  Saved {out_xml} in {time.time()-t0:.1f} s")
    return out_xml

# Run blastp.
blast_xml = os.path.join(WORK_DIR, "blastp_human_betaglobin_vs_swissprot.xml")
cached_qblast("blastp", DATABASE, query_record.seq, blast_xml,
              expect=10, hitlist_size=HITLIST_SIZE)

In [ ]:
# Parse the XML and build a tidy DataFrame of hits.
import re

with open(blast_xml) as f:
    blast_record = NCBIXML.read(f)

rows = []
for alignment in blast_record.alignments:
    # NCBI BLAST against swissprot returns RefSeq-style titles:
    #   sp|P02024.2| RecName: Full=Hemoglobin subunit beta; ... [Gorilla gorilla gorilla]
    # Species is the [last bracketed group]; the leading sp|ACCN.ver| is the entry.
    title = alignment.hit_def or alignment.title
    species = ""
    for m in re.finditer(r"\[([^\[\]]+)\]", title):
        species = m.group(1)              # take the LAST bracketed organism name
    m_acc   = re.match(r"(?:sp|tr)\|([A-Z0-9]+)", alignment.hit_id or alignment.title)
    accession = m_acc.group(1) if m_acc else alignment.accession
    entry     = ""                         # NCBI doesn't supply UniProt entry names

    best_hsp = min(alignment.hsps, key=lambda h: h.expect)
    rows.append({
        "accession":   accession,
        "entry_name":  entry,
        "species":     species,
        "subject_len": alignment.length,
        "evalue":      best_hsp.expect,
        "bit_score":   best_hsp.bits,
        "identity":    best_hsp.identities / best_hsp.align_length,
        "align_len":   best_hsp.align_length,
        "qcover":      best_hsp.align_length / blast_record.query_length,
    })

hits = pd.DataFrame(rows).sort_values("evalue").reset_index(drop=True)
print(f"Got {len(hits)} hits.\n")
hits.head(15)

In [ ]:
# Visualise the E-value and bit-score distributions, with the cutoff line.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax1, ax2 = axes

# E-value: -log10 (replace zeros with a floor so log is finite)
log_e = -np.log10(hits["evalue"].clip(lower=1e-200))
ax1.hist(log_e, bins=30, color="#2a7f8a", edgecolor="white")
ax1.axvline(-np.log10(EVALUE_CUT), color="#b8602f", linestyle="--",
            label=f"cutoff E = {EVALUE_CUT:g}")
ax1.set_xlabel(r"$-\log_{10}(E)$")
ax1.set_ylabel("Number of hits")
ax1.set_title("E-value distribution")
ax1.legend()

ax2.hist(hits["bit_score"], bins=30, color="#2a7f8a", edgecolor="white")
ax2.set_xlabel("Bit score")
ax2.set_ylabel("Number of hits")
ax2.set_title("Bit-score distribution")

fig.tight_layout()
plt.show()

**Question 1:** Why filter by E-value rather than percent identity? When *would* percent identity be the better filter?

*Answer hint:* E-value is database-size aware: a 30 % identity hit means very different things in SwissProt (~570k entries) vs. nr (~600M entries). Percent identity is fine for closely-related sequences (e.g., orthologs from the same phylum) where you have an a priori expectation of similarity. For divergent searches across kingdoms — exactly the kind that pull in leghemoglobin from a vertebrate query — E-value is the safer threshold.

## 3. Inside an HSP

For the top hit, let's look at the actual alignment block — query positions, subject positions, identities, the alignment string. This is the *high-scoring segment pair* (HSP) that BLAST built by extending a word seed.

For an interactive demo of word seeding parameters W and T, open `docs/fasta-blast.html` in a browser.

In [ ]:
# Show the top HSP for the most significant hit.
top = blast_record.alignments[0]
hsp = min(top.hsps, key=lambda h: h.expect)

print(f"Top hit: {top.title[:90]}")
print(f"  E-value     : {hsp.expect:.2e}")
print(f"  Bit score   : {hsp.bits:.1f}")
print(f"  Identity    : {hsp.identities}/{hsp.align_length}  ({100*hsp.identities/hsp.align_length:.1f} %)")
print(f"  Positives   : {hsp.positives}/{hsp.align_length}  ({100*hsp.positives/hsp.align_length:.1f} %)")
print(f"  Query  range: {hsp.query_start}..{hsp.query_end}")
print(f"  Sbjct  range: {hsp.sbjct_start}..{hsp.sbjct_end}")
print()

# Pretty-print the alignment in 60-column blocks.
def print_block(qry, mid, sbj, qstart, sstart, width=60):
    for i in range(0, len(qry), width):
        q = qry[i:i+width]; m = mid[i:i+width]; s = sbj[i:i+width]
        # count non-gap residues consumed in this block to recompute position labels
        qend = qstart + len(q.replace("-", "")) - 1
        send = sstart + len(s.replace("-", "")) - 1
        print(f"Q  {qstart:4d}  {q}  {qend}")
        print(f"          {m}")
        print(f"S  {sstart:4d}  {s}  {send}\n")
        qstart = qend + 1
        sstart = send + 1

print_block(hsp.query, hsp.match, hsp.sbjct, hsp.query_start, hsp.sbjct_start)

**Question 2:** Find a hit somewhere around 30% identity (these are usually the invertebrate or plant globins). Read its HSP. Are most of the substitutions *conservative* (BLOSUM62-positive, marked with `+` in the match string), or are they roughly random?

*Answer hint:* Even at low identity, the substitutions in a true homolog cluster around chemically similar amino acids — Leu↔Ile, Lys↔Arg, Asp↔Glu — because the protein still has to fold. Random substitutions have no such bias. The match string's `+` count is a quick visual proxy.

## 4. Filtering for downstream MSA

We want a clean set of globin homologs to feed `5_2_msa.ipynb`. The rule:

1. $E < 10^{-3}$ (homology, including the twilight zone).
2. Alignment length $> 0.5 \times$ query length (avoid fragments).
3. **One hit per species** (deduplicate organism — paralogs blur a species tree).
4. Always keep the query itself, and always keep the leghemoglobin hit if present (we want it as outgroup for `5_3_phylogeny.ipynb`).

In [ ]:
selected = hits[
    (hits["evalue"] < EVALUE_CUT) &
    (hits["align_len"] > 0.5 * blast_record.query_length)
].copy()

# One hit per species — keep the lowest-E representative.
deduped = selected.sort_values("evalue").drop_duplicates(subset="species", keep="first")

# Cap at MAX_FOR_MSA: pick a *diverse* slice rather than the top-N by E-value
# (which would all be near-identical mammalian β-globins). Sample evenly across
# the bit-score range so the downstream MSA spans some real evolutionary depth.
if len(deduped) > MAX_FOR_MSA:
    sorted_by_score = deduped.sort_values("bit_score", ascending=False).reset_index(drop=True)
    idx = np.linspace(0, len(sorted_by_score) - 1, MAX_FOR_MSA).round().astype(int)
    diverse = sorted_by_score.iloc[idx].drop_duplicates(subset="accession")
else:
    diverse = deduped

print(f"After E < {EVALUE_CUT:g} and length filter: {len(selected)} hits")
print(f"After species dedup:                     {len(deduped)} hits")
print(f"After diversity downsample to {MAX_FOR_MSA}:        {len(diverse)} hits\n")
diverse.head(20)

**Question 3:** Two of the top BLAST hits are paralogs from the same species (e.g. human β-globin and human α-globin). Should you keep *both* in your MSA? It depends on what you want the downstream phylogeny to *show*.

*Answer hint:* For a **species tree** (when did human and chicken last share a common ancestor?), keep one ortholog per species. For a **gene-family tree** (how did α and β globins diverge from a common ancestral globin?), keep paralogs and label them. The deduplication rule above is conservative; the term project may need to relax it.

In [ ]:
# Write the selected hits as a multi-FASTA file. We need to fetch their full
# sequences from UniProt; the BLAST XML contains the alignment but not the
# full subject sequence.
selected_fa = os.path.join(WORK_DIR, "selected_globins.fa")
n_written = 0
records = []

for _, row in diverse.iterrows():
    acc = row["accession"]
    if not acc:
        continue
    cache_path = os.path.join(WORK_DIR, f"{acc}.fasta")
    if not os.path.exists(cache_path):
        try:
            url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
            req = urllib.request.Request(url, headers={"User-Agent": f"gtubeng/5_1_blast {NCBI_EMAIL}"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                with open(cache_path, "wb") as g:
                    g.write(resp.read())
            time.sleep(1)  # one second between UniProt fetches
        except Exception as e:
            print(f"  ⚠️  Could not fetch {acc}: {e}")
            continue
    rec = SeqIO.read(cache_path, "fasta")
    records.append(rec)
    n_written += 1

with open(selected_fa, "w") as g:
    for r in records:
        g.write(f">{r.id} {r.description}\n{str(r.seq)}\n")

print(f"Wrote {selected_fa} ({n_written} sequences, {os.path.getsize(selected_fa)} bytes)")
print(f"\n→ This file is the input to 5_2_msa.ipynb.")

# Also save a CSV summary of the diverse-downsampled hits.
summary_csv = os.path.join(WORK_DIR, "blast_summary.csv")
diverse.to_csv(summary_csv, index=False)
print(f"Wrote {summary_csv}")

**A note on what plain `blastp` reaches.** Even with `HITLIST_SIZE=100`, the hits returned for a human β-globin query against SwissProt are *all* vertebrate hemoglobins. Plant leghemoglobin and invertebrate globins (the kind that would give `5_3_phylogeny.ipynb` a clean outgroup) are too divergent — their E-values fall outside the search horizon at default sensitivity. This is *not* a bug in the notebook; it is the empirical reach of position-independent scoring with BLOSUM62. Two ways to find them:

- **PSI-BLAST iterations** (next section) — builds a profile from the first round and uses it to detect divergent homologs.
- **HMMER profile search** (`6_3_hmmer_search.ipynb`) — same idea, more sensitive, with calibrated statistics.

When `5_3_phylogeny.ipynb` cannot find leghemoglobin in its input, it falls back to midpoint rooting and tells you so.

## 5. PSI-BLAST: an honest look at what `service='psi'` actually does

`blastp` builds a query–database alignment using a static substitution matrix (BLOSUM62). At the twilight zone (~25 % identity), this misses real homologs simply because BLOSUM62 cannot tell signal from noise on so few conserved residues.

**PSI-BLAST** — Position-Specific Iterated BLAST — fixes this:

1. Round 1: ordinary `blastp`, take all hits with $E < 0.005$.
2. Build a **PSSM** (position-specific scoring matrix) from those hits: each column gets its own substitution weights.
3. Round 2: search again with the PSSM as the query.
4. Repeat until convergence (no new hits) or a fixed number of iterations.

**A real-world API gotcha worth knowing.** The Biopython call `NCBIWWW.qblast(..., service="psi")` returns **only round 1**. The NCBI BLAST web UI exposes an interactive *Iterate* button between rounds; that button has no programmatic equivalent. So the cells below will show that `service='psi'` returns the same hits as plain `blastp` — by design, not by bug. To genuinely iterate you have three options:

- **NCBI BLAST web UI** — point and click; not scriptable.
- **Local BLAST+** — `psiblast -num_iterations 3 -query q.fa -db swissprot -out_pssm pssm.smp`. Requires installing BLAST+ locally.
- **HMMER `jackhmmer`** — iterates server-side via EBI's REST tools, no local install. We use this in `6_3_hmmer_search.ipynb` (see `hmmer_search_work/legh_jackhmmer.tbl`) and it is the cleanest route in this course.

PSI-BLAST has one quirky requirement when called via `qblast`: `word_size=3` (the default for `blastp` is 3 already, but the API needs it explicit when `service="psi"`).

In [ ]:
psi_xml = os.path.join(WORK_DIR, "psi_blast_human_betaglobin.xml")
cached_qblast("blastp", DATABASE, query_record.seq, psi_xml,
              expect=10, hitlist_size=HITLIST_SIZE,
              service="psi", word_size=3)

with open(psi_xml) as f:
    psi_record = NCBIXML.read(f)

# Same regex shape as the plain-BLAST cell above. NCBI titles look like
# 'sp|P68871.2| RecName: ...' — note the version (.2) between the accession
# and the second pipe, so we anchor on the alphanumeric run only.
psi_hits = []
for a in psi_record.alignments:
    m_acc = re.match(r"(?:sp|tr)\|([A-Z0-9]+)", a.hit_id or a.title)
    if m_acc:
        psi_hits.append((m_acc.group(1), min(h.expect for h in a.hsps)))

psi_df = pd.DataFrame(psi_hits, columns=["accession", "evalue_psi"])
print(f"PSI-BLAST returned {len(psi_df)} hits  (round 1 only — see markdown above).")

In [ ]:
# Sanity check: with no PSSM uploaded, round-1 PSI-BLAST is essentially the
# same search as plain blastp. In practice the PSI endpoint returns a *superset*
# — your plain top-N plus a tail of weaker hits that PSSM construction would
# normally use as seed alignments. Every plain hit should appear in the PSI
# set. If the overlap is small, something has changed in the API.
plain_set = set(hits["accession"])
psi_set   = set(psi_df["accession"])

in_both    = plain_set & psi_set
only_plain = plain_set - psi_set
only_psi   = psi_set - plain_set
print(f"Plain blastp hits      : {len(plain_set)}")
print(f"PSI-BLAST round-1 hits : {len(psi_set)}")
print(f"Plain ∩ PSI            : {len(in_both)}  ({100*len(in_both)/max(len(plain_set), 1):.1f} % of plain)")
print(f"Only in plain          : {len(only_plain)}")
print(f"Only in PSI (weaker)   : {len(only_psi)}")
print()
print("Note: 'only in PSI' hits are not new homologs — they are weaker hits the")
print("PSI endpoint retains for hypothetical PSSM seeding. To genuinely iterate and")
print("reach divergent homologs, see jackhmmer in 6_3_hmmer_search.ipynb.")

**Question 4:** Why does the web `service='psi'` return essentially the same hits as plain `blastp` here? What would you actually need to install or run to demonstrate iterative profile search? Which course tool already does this for you, and on what kind of query (similar to β-globin, or very different) would iteration help most?

*Answer hint:* `qblast(service="psi")` only submits round 1; the NCBI web UI's *Iterate* button has no programmatic equivalent. To iterate from code you would either install BLAST+ locally (`psiblast -num_iterations N`) or use HMMER's `jackhmmer`, which iterates server-side via EBI and is the route used in `6_3_hmmer_search.ipynb`. Iteration matters most when the query is *itself in the twilight zone* relative to the family — e.g., trying to recover plant leghemoglobin from an animal Mb query, where round 1 of `blastp` returns nothing and only profile-based iteration brings the plant hits in (compare `hmmer_search_work/legh_jackhmmer.tbl`).

## 6. When BLAST is *not* the right tool

BLAST excels at finding sequences that are still recognisably similar to your query. When the signal is buried — sub-30 % identity, fragmented domains, or you want to find *all* members of a family at once — you need a different tool:

- **HMMER profile search** (`6_2_hmmbuild.ipynb`, `6_3_hmmer_search.ipynb`) — builds a position-specific model from an MSA, finds remote homologs more reliably than PSI-BLAST.
- **Structural alignment** (e.g. Foldseek, DALI) — compares 3D folds; works when sequence has decayed below detectable similarity but the fold is preserved.
- **Pfam / InterProScan** — precomputed family models; tells you "this region is a globin domain" without you having to build the model.

The term project asks you to combine BLAST, HMMER, and Pfam exactly because no single tool catches everything.

## Resources

| Topic | Reference |
|-------|-----------|
| BLAST original | Altschul SF et al. (1990). *J Mol Biol* 215:403. |
| Gapped BLAST + PSI-BLAST | Altschul SF et al. (1997). *Nucleic Acids Res* 25:3389. |
| Karlin–Altschul statistics | Karlin & Altschul (1990). *PNAS* 87:2264. |
| NCBI BLAST web docs | https://blast.ncbi.nlm.nih.gov/doc/blast-help/ |
| BioPython BLAST tutorial | https://biopython.org/docs/latest/Tutorial/chapter_blast.html |
| EBI Job Dispatcher | Madeira F et al. (2024). *Nucleic Acids Res* 52:W521. |
| Course widget — word seeding & E-value | `docs/fasta-blast.html` |
| Course widget — substitution matrices | `docs/scoring-matrices.html` |
| Course widget — HMMER (alternative) | `docs/hmm-hmmer.html` |
| Iterative profile search that actually iterates | `6_3_hmmer_search.ipynb` (`jackhmmer` via EBI REST) |
| Local PSI-BLAST iteration (if you install BLAST+) | `psiblast -num_iterations 3 -query q.fa -db swissprot` |
| Next notebook | `5_2_msa.ipynb` (consumes `blast_work/selected_globins.fa`) |